# 05 Candidate Cluster Profiling

This notebook profiles a small set of candidate K-Means solutions before choosing a final segmentation model.

Cluster labels are created only in memory for interpretation. No final customer-cluster CSV is created.


## Purpose of this notebook

The previous notebook compared baseline K-Means metrics for Baseline A and Baseline B. Metrics alone are not enough for a good university project defense: the clusters also need to be understandable.

This notebook profiles selected candidate solutions and recommends which one or two candidates should be investigated further. It does not select a final model.


## Setup and data loading

This section imports the project helpers, reusable modeling utilities, and raw datasets. The fixed reference date keeps age and tenure reproducible.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def has_raw_datasets(candidate):
    root_layout = (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists()
    project_files_layout = (candidate / "Project files" / "customer_info.csv").exists() and (candidate / "Project files" / "customer_basket.csv").exists()
    return root_layout or project_files_layout


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if has_raw_datasets(candidate):
            return candidate
    raise FileNotFoundError("Could not locate raw datasets at the repository root or in Project files/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE_DATE = pd.Timestamp("2026-05-30")
RANDOM_STATE = 42

print(f"Project root: {PROJECT_ROOT}")
print(f"Reference date for age and tenure: {REFERENCE_DATE.date()}")
print(f"Random state: {RANDOM_STATE}")


In [ ]:
from src.data_loading import load_datasets
from src.features import build_customer_feature_table
from src.modeling import (
    add_cluster_labels,
    compare_cluster_profiles_to_global,
    fit_kmeans_labels,
    prepare_modeling_matrix,
    profile_clusters,
    summarize_cluster_sizes,
)


In [ ]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

print(f"customer_info rows: {len(customer_info):,}")
print(f"unique customers in customer_info: {customer_info['customer_id'].nunique():,}")
print(f"customer_basket rows: {len(customer_basket):,}")


## Build and validate feature table

The feature table is rebuilt with the existing feature-engineering workflow. The checks confirm that the same clean table is available and that no final clustering output already exists.


In [ ]:
feature_table, metadata = build_customer_feature_table(
    customer_info,
    customer_basket,
    reference_date=REFERENCE_DATE,
)

outputs_dir = PROJECT_ROOT / "outputs"
final_output_files = []
if outputs_dir.exists():
    for path in outputs_dir.iterdir():
        name = path.name.lower()
        looks_like_final_cluster_output = path.suffix.lower() == ".csv" and (
            "cluster" in name or "segment" in name
        )
        if path.is_file() and looks_like_final_cluster_output:
            final_output_files.append(path.name)

initial_validation = pd.DataFrame(
    [
        {"check": "feature table shape is 33038 x 77", "value": str(feature_table.shape), "passes": feature_table.shape == (33038, 77)},
        {"check": "customer_id is unique", "value": feature_table["customer_id"].is_unique, "passes": feature_table["customer_id"].is_unique},
        {"check": "feature table has no missing values", "value": int(feature_table.isna().sum().sum()), "passes": int(feature_table.isna().sum().sum()) == 0},
        {"check": "customer_name is not present", "value": "customer_name" in feature_table.columns, "passes": "customer_name" not in feature_table.columns},
        {"check": "no final clustering csv outputs", "value": final_output_files, "passes": len(final_output_files) == 0},
    ]
)

display(initial_validation)
assert initial_validation["passes"].all()


## Candidate solutions to profile

The candidates are selected from the baseline comparison notebook. They include simple and more detailed solutions from Baseline A, plus two Baseline B candidates to check whether basket features help or dominate the interpretation.


In [ ]:
candidate_solutions = [
    {"candidate": "A_k2", "feature_set": "Baseline A", "k": 2},
    {"candidate": "A_k4", "feature_set": "Baseline A", "k": 4},
    {"candidate": "A_k6", "feature_set": "Baseline A", "k": 6},
    {"candidate": "B_k4", "feature_set": "Baseline B", "k": 4},
    {"candidate": "B_k6", "feature_set": "Baseline B", "k": 6},
]

display(pd.DataFrame(candidate_solutions))


## Reusable modeling workflow

The modeling utility module keeps repeated clustering steps simple: transform selected skewed columns, scale the matrix, fit deterministic K-Means labels, and profile clusters in memory.


In [ ]:
def existing(columns):
    return [column for column in columns if column in feature_table.columns]

spend_share_columns = [column for column in feature_table.columns if column.startswith("spend_share_")]

model_features_a_no_basket = existing(
    [
        "customer_age",
        "customer_tenure_years",
        "kids_home",
        "teens_home",
        "distinct_stores_visited",
        "lifetime_total_distinct_products",
        "has_loyalty_card",
        "number_complaints",
        "promotion_pct_clean",
        "total_lifetime_spend",
        *spend_share_columns,
    ]
)

reduced_basket_features = ["has_sampled_basket", "basket_count", "avg_basket_size"]
model_features_b_with_basket = model_features_a_no_basket + existing(reduced_basket_features)

feature_sets = {
    "Baseline A": {
        "features": model_features_a_no_basket,
        "log_columns": ["total_lifetime_spend"],
    },
    "Baseline B": {
        "features": model_features_b_with_basket,
        "log_columns": ["total_lifetime_spend", "basket_count"],
    },
}

all_basket_feature_columns = [
    "has_sampled_basket",
    "basket_count",
    "avg_basket_size",
    "median_basket_size",
    "max_basket_size",
    "total_basket_items",
    "unique_basket_products",
]

feature_set_validation = pd.DataFrame(
    [
        {"check": "all Baseline A features exist", "value": len(model_features_a_no_basket), "passes": all(column in feature_table.columns for column in model_features_a_no_basket)},
        {"check": "all Baseline B features exist", "value": len(model_features_b_with_basket), "passes": all(column in feature_table.columns for column in model_features_b_with_basket)},
        {"check": "customer_id not used in Baseline A", "value": "customer_id" in model_features_a_no_basket, "passes": "customer_id" not in model_features_a_no_basket},
        {"check": "customer_id not used in Baseline B", "value": "customer_id" in model_features_b_with_basket, "passes": "customer_id" not in model_features_b_with_basket},
        {"check": "Baseline A has no basket features", "value": sorted(set(model_features_a_no_basket) & set(all_basket_feature_columns)), "passes": set(model_features_a_no_basket).isdisjoint(all_basket_feature_columns)},
        {"check": "Baseline B has only reduced basket features", "value": sorted(set(model_features_b_with_basket) & set(all_basket_feature_columns)), "passes": set(model_features_b_with_basket) & set(all_basket_feature_columns) == set(reduced_basket_features)},
    ]
)

display(feature_set_validation)
assert feature_set_validation["passes"].all()


## Fit candidate cluster solutions

Each candidate is fitted with `RANDOM_STATE = 42`. Labels are added only to dataframe copies inside this notebook for profiling.


In [ ]:
prepared_matrices = {}
for feature_set_name, config in feature_sets.items():
    transformed_data, X_scaled, scaler = prepare_modeling_matrix(
        feature_table,
        config["features"],
        log_columns=config["log_columns"],
    )
    prepared_matrices[feature_set_name] = {
        "transformed_data": transformed_data,
        "X_scaled": X_scaled,
        "scaler": scaler,
    }

candidate_results = {}
for solution in candidate_solutions:
    feature_set_name = solution["feature_set"]
    candidate_name = solution["candidate"]
    k = solution["k"]
    X_scaled = prepared_matrices[feature_set_name]["X_scaled"]
    labels = fit_kmeans_labels(X_scaled, k, random_state=RANDOM_STATE)

    labeled_table = add_cluster_labels(
        feature_table,
        labels,
        cluster_column="cluster",
    )
    candidate_results[candidate_name] = {
        "feature_set": feature_set_name,
        "k": k,
        "labels": labels,
        "labeled_table": labeled_table,
    }

fit_summary = pd.DataFrame(
    [
        {
            "candidate": candidate_name,
            "feature_set": result["feature_set"],
            "k": result["k"],
            "label_count": len(result["labels"]),
            "unique_clusters": len(np.unique(result["labels"])),
            "random_state": RANDOM_STATE,
        }
        for candidate_name, result in candidate_results.items()
    ]
)

display(fit_summary)


## Cluster size comparison

Cluster sizes help identify solutions that are too simple, too fragmented, or difficult to defend because one cluster is tiny or one cluster dominates the solution.


In [ ]:
cluster_size_tables = {}
cluster_size_rows = []

for candidate_name, result in candidate_results.items():
    size_table = summarize_cluster_sizes(result["labels"])
    size_table.insert(0, "candidate", candidate_name)
    size_table.insert(1, "feature_set", result["feature_set"])
    size_table.insert(2, "k", result["k"])
    cluster_size_tables[candidate_name] = size_table
    cluster_size_rows.append(size_table)

cluster_size_summary = pd.concat(cluster_size_rows, ignore_index=True)
cluster_size_display = cluster_size_summary.copy()
cluster_size_display["cluster_percentage"] = (cluster_size_display["cluster_percentage"] * 100).round(2)

display(cluster_size_display)


## Cluster profile tables

The profile columns combine modeling features and interpretation-only features. The displayed tables keep the most important columns readable, while the full profile tables remain available in memory.


In [ ]:
core_profile_columns = existing(
    [
        "customer_age",
        "customer_tenure_years",
        "kids_home",
        "teens_home",
        "total_children_home",
        "distinct_stores_visited",
        "lifetime_total_distinct_products",
        "total_lifetime_spend",
        "has_loyalty_card",
        "number_complaints",
        "promotion_pct_clean",
    ]
)

basket_profile_columns = existing(
    [
        "has_sampled_basket",
        "basket_count",
        "avg_basket_size",
        "unique_basket_products",
        "total_basket_items",
    ]
)

demographic_profile_columns = existing(
    [
        "gender_female",
        "gender_male",
        "gender_unknown",
        "degree_bsc",
        "degree_msc",
        "degree_phd",
        "degree_unknown",
    ]
)

data_quality_profile_columns = existing(
    [
        "promotion_pct_suspicious",
        "first_transaction_year_suspicious",
        "customer_age_missing_or_invalid",
        "customer_tenure_missing_or_invalid",
        *[column for column in feature_table.columns if column.endswith("_was_missing")],
    ]
)

profile_columns = existing(
    core_profile_columns
    + spend_share_columns
    + basket_profile_columns
    + demographic_profile_columns
    + data_quality_profile_columns
)

profile_display_columns = ["customer_count"] + existing(
    [
        "customer_age",
        "customer_tenure_years",
        "total_children_home",
        "total_lifetime_spend",
        "spend_share_groceries",
        "spend_share_electronics",
        "spend_share_meat",
        "has_loyalty_card",
        "number_complaints",
        "promotion_pct_clean",
        "has_sampled_basket",
        "basket_count",
        "avg_basket_size",
        "unique_basket_products",
        "gender_female",
        "degree_unknown",
        "promotion_pct_suspicious",
    ]
)

cluster_profiles = {}
for candidate_name, result in candidate_results.items():
    profile = profile_clusters(result["labeled_table"], "cluster", profile_columns)
    cluster_profiles[candidate_name] = profile
    print(f"Profile table for {candidate_name}")
    display(profile[["cluster", *profile_display_columns]].round(3))


## Differences from global average

The next tables show which variables most distinguish each cluster from the global average. This is useful for interpretation because it highlights what makes each cluster different.


In [ ]:
cluster_global_differences = {}
strongest_difference_rows = []

for candidate_name, result in candidate_results.items():
    comparison = compare_cluster_profiles_to_global(
        result["labeled_table"],
        "cluster",
        profile_columns,
    )
    cluster_global_differences[candidate_name] = comparison

    for _, row in comparison.iterrows():
        cluster_id = int(row["cluster"])
        differences = row[profile_columns].abs().sort_values(ascending=False).head(6)
        for feature_name, abs_difference in differences.items():
            strongest_difference_rows.append(
                {
                    "candidate": candidate_name,
                    "cluster": cluster_id,
                    "feature": feature_name,
                    "difference_from_global": row[feature_name],
                    "abs_difference": abs_difference,
                }
            )

strongest_differences = pd.DataFrame(strongest_difference_rows)
strongest_differences["difference_from_global"] = strongest_differences["difference_from_global"].round(3)
strongest_differences["abs_difference"] = strongest_differences["abs_difference"].round(3)

display(strongest_differences)


## Basket availability diagnostic

For Baseline B candidates, we check whether clusters are mainly separated by basket availability and basket activity. If that happens, the solution may be too basket-driven because basket data is sampled.


In [ ]:
basket_diagnostics = {}
for candidate_name, result in candidate_results.items():
    if result["feature_set"] != "Baseline B":
        continue

    diagnostic = result["labeled_table"].groupby("cluster").agg(
        customers=("customer_id", "count"),
        sampled_basket_share=("has_sampled_basket", "mean"),
        avg_basket_count=("basket_count", "mean"),
        avg_basket_size=("avg_basket_size", "mean"),
        avg_unique_basket_products=("unique_basket_products", "mean"),
        avg_total_basket_items=("total_basket_items", "mean"),
        avg_total_lifetime_spend=("total_lifetime_spend", "mean"),
    ).reset_index()
    diagnostic["customer_percentage"] = diagnostic["customers"] / len(feature_table) * 100
    basket_diagnostics[candidate_name] = diagnostic.round(3)

    print(f"Basket diagnostic for {candidate_name}")
    display(basket_diagnostics[candidate_name])


### Basket diagnostic interpretation

The Baseline B candidates should be treated carefully. If a cluster mainly differs by `has_sampled_basket`, `basket_count`, or `avg_basket_size`, it may be more about transaction sampling than a stable customer segment. If basket features help separate customers while still showing meaningful spend and demographic differences, Baseline B may be promising for interpretation.


## Candidate-by-candidate interpretation notes

These notes are cautious. They summarize what each candidate is useful for, without selecting a final model.


In [ ]:
interpretation_notes = pd.DataFrame(
    [
        {
            "candidate": "A_k2",
            "note": "Very simple and easy to explain, but may be too simple for a final segmentation.",
        },
        {
            "candidate": "A_k4",
            "note": "Candidate to investigate further because it may balance simplicity and richer interpretation without basket dependence.",
        },
        {
            "candidate": "A_k6",
            "note": "Promising for interpretation if the six groups remain distinct and not too fragmented.",
        },
        {
            "candidate": "B_k4",
            "note": "Candidate to investigate further, but check whether basket availability drives the result too strongly.",
        },
        {
            "candidate": "B_k6",
            "note": "May provide more detail, but may also become harder to defend or too basket-driven.",
        },
    ]
)

display(interpretation_notes)


## Recommendation for next phase

At this stage, no final model is selected. Based on interpretability needs, the strongest candidates to investigate further are:

- `A_k4` or `A_k6`, because they avoid basket sampling dependence and may still produce interpretable customer groups.
- `B_k4`, if the basket diagnostic shows that basket features add useful behavior information without dominating the solution.

The next phase should compare these candidates in more detail and only then choose one final model configuration.


## Final validation checks

These checks confirm that the notebook ran the requested candidate profiling workflow without creating final clustering outputs.


In [ ]:
outputs_dir = PROJECT_ROOT / "outputs"
final_output_files = []
if outputs_dir.exists():
    for path in outputs_dir.iterdir():
        name = path.name.lower()
        looks_like_final_cluster_output = path.suffix.lower() == ".csv" and (
            "cluster" in name or "segment" in name
        )
        if path.is_file() and looks_like_final_cluster_output:
            final_output_files.append(path.name)

expected_candidates = {"A_k2", "A_k4", "A_k6", "B_k4", "B_k6"}
final_validation = pd.DataFrame(
    [
        {"check": "feature table shape remains 33038 x 77", "value": str(feature_table.shape), "passes": feature_table.shape == (33038, 77)},
        {"check": "customer_id is unique", "value": feature_table["customer_id"].is_unique, "passes": feature_table["customer_id"].is_unique},
        {"check": "feature table has no missing values", "value": int(feature_table.isna().sum().sum()), "passes": int(feature_table.isna().sum().sum()) == 0},
        {"check": "customer_id is not a modeling feature", "value": "customer_id" in model_features_a_no_basket or "customer_id" in model_features_b_with_basket, "passes": "customer_id" not in model_features_a_no_basket and "customer_id" not in model_features_b_with_basket},
        {"check": "all expected candidates fitted", "value": sorted(candidate_results.keys()), "passes": set(candidate_results.keys()) == expected_candidates},
        {"check": "all candidates used RANDOM_STATE 42", "value": RANDOM_STATE, "passes": RANDOM_STATE == 42},
        {"check": "cluster size summaries created", "value": cluster_size_summary.shape, "passes": len(cluster_size_summary) > 0},
        {"check": "cluster profile tables created", "value": len(cluster_profiles), "passes": set(cluster_profiles.keys()) == expected_candidates},
        {"check": "Baseline B basket diagnostics created", "value": sorted(basket_diagnostics.keys()), "passes": set(basket_diagnostics.keys()) == {"B_k4", "B_k6"}},
        {"check": "no final clustering csv outputs", "value": final_output_files, "passes": len(final_output_files) == 0},
    ]
)

display(final_validation)
assert final_validation["passes"].all()
print("Candidate cluster profiling validation passed. No final model was selected and no final cluster CSV was created.")
